# Limpeza de Dados


In [1]:
#importando bibliotecas
from haversine import haversine 
import pandas as pd
import re as re
import plotly.express as px
import folium 
import plotly.graph_objects as go   
import numpy as np


#importando a base de dados
df = pd.read_csv('../dataset/train.csv')
df1 = df.copy()


# Remover os espaços das strings
df1.loc[:,'ID'] = df1.loc[:, 'ID'].str.strip()
df1.loc[:,'Road_traffic_density'] = df1.loc[:, 'Road_traffic_density'].str.strip()
df1.loc[:,'Type_of_order'] = df1.loc[:, 'Type_of_order'].str.strip()
df1.loc[:,'Type_of_vehicle'] = df1.loc[:, 'Type_of_vehicle'].str.strip()
df1.loc[:,'Festival'] = df1.loc[:, 'Festival'].str.strip()
df1.loc[:,'City'] = df1.loc[:, 'City'].str.strip()


# Excluir as linhas com a idade dos entregadores vazia
linhas_vazias = df1['Delivery_person_Age'] != 'NaN '
df1 = df1.loc[linhas_vazias, :]
linhas_vazias = df1['Delivery_person_Age'] != 'NaN '
df1= df1.loc[linhas_vazias, :]
linhas_vazias = df1['City'] != 'NaN'
df1= df1.loc[linhas_vazias, :]
linhas_vazias = df1['Road_traffic_density'] != 'NaN'
df1= df1.loc[linhas_vazias, :]
linhas_vazias = df1['Festival'] != 'NaN'
df1 = df1.loc[linhas_vazias, :]

# Conversao de texto/categoria/string para numeros inteiros
df1['Delivery_person_Age'] = df1['Delivery_person_Age'].astype( int )


# Conversao de texto/categoria/strings para numeros decimais
df1['Delivery_person_Ratings'] = df1['Delivery_person_Ratings'].astype( float )

# Conversao de texto para data
df1['Order_Date'] = pd.to_datetime( df1['Order_Date'], format='%d-%m-%Y' )
linhas_vazias = df1['multiple_deliveries'] != 'NaN '
df1 = df1.loc[linhas_vazias, :]
df1['multiple_deliveries'] = df1['multiple_deliveries'].astype( int )



# Comando para remover o texto de números
df1['Time_taken(min)'] = df1['Time_taken(min)'].apply(lambda x: x.split ( '(min)')[1])
df1['Time_taken(min)'] = df1['Time_taken(min)'].astype(int)

# 1.0 Visão De Epresa:

## Quantidade de pedidos por dia.


In [2]:
colums_01 = ['ID','Order_Date']
df_qtd_pedidos_dia = df1.loc[:, colums_01].groupby(['Order_Date']).count().reset_index()

px.bar(df_qtd_pedidos_dia, x='Order_Date', y= 'ID')

## Quantidade de pedidos por semana.

In [3]:
df1['week_of_year'] = df1['Order_Date'].dt.strftime('%U')
colums_02 = ['ID','week_of_year']
df_qtd_pedidos_semana = df1.loc[:, colums_02].groupby(['week_of_year']).count().reset_index()
px.line(df_qtd_pedidos_semana, x='week_of_year', y='ID')


## Distribuição dos pedidos por tipo de tráfego.

In [4]:
columns_03 = ['ID','Road_traffic_density']

df_pedidos_trafego = df1.loc[:,columns_03].groupby(['Road_traffic_density']).count().reset_index()
df_pedidos_trafego['entregas_perc'] = df_pedidos_trafego['ID']/df_pedidos_trafego['ID'].sum()

px.pie(df_pedidos_trafego, values='entregas_perc', names='Road_traffic_density')

## Comparação do volume de pedidos por cidade e tipo de tráfego.

In [5]:
colums_04 = ['ID','Road_traffic_density','City']
pedidos_cidade_trafego = df1.loc[:, colums_04].groupby(['City', 'Road_traffic_density']).count().reset_index()
pedidos_cidade_trafego = pedidos_cidade_trafego.loc[(pedidos_cidade_trafego['City']!= 'NaN') &(pedidos_cidade_trafego['Road_traffic_density']!= 'NaN'), :]

px.scatter(pedidos_cidade_trafego, x='City', y='Road_traffic_density', size='ID')

## A quantidade de pedidos por entregador por semana.

In [6]:
df_aux01 = df1.loc[:,['ID', 'week_of_year']].groupby(['week_of_year']).count().reset_index()
df_aux02 = df1.loc[:,['Delivery_person_ID', 'week_of_year']].groupby(['week_of_year']).nunique().reset_index()

df_aux = pd.merge(df_aux01, df_aux02, how='inner')

df_aux['qtd_pedidos_entregador'] = df_aux['ID']/df_aux['Delivery_person_ID']

px.line(df_aux,x='week_of_year', y='qtd_pedidos_entregador')

## A localização central de cada cidade por tipo de tráfego

In [7]:
columns = ['City', 'Road_traffic_density', 'Delivery_location_latitude', 'Delivery_location_longitude']
df_aux = df1.loc[:, columns].groupby(['City', 'Road_traffic_density']).median().reset_index()
df_aux =df_aux.loc[df_aux['City'] != 'NaN']
df_aux =df_aux.loc[df_aux['Road_traffic_density'] != 'NaN']
df_aux = df_aux.head()
map = folium.Map()

for index, location_info in df_aux.iterrows():
    folium.Marker([location_info['Delivery_location_latitude'],
    location_info['Delivery_location_longitude']],
    popup = location_info[['City','Road_traffic_density']]).add_to(map)
    
map

# 2.0 Visão dos entregadores


## A menor e maior idade dos entregadores.

In [8]:
max_age = df1.loc[:, 'Delivery_person_Age'].max()
min_age = df1.loc[:, 'Delivery_person_Age'].min()

print(f'A Maior idade é:{max_age}')
print(f'A Menor idade é:{max_age}')

A Maior idade é:39
A Menor idade é:39


## A pior e a melhor condição de veículos.

In [9]:
melhor_condicao_veiculo = df.loc[:, 'Vehicle_condition'].max()
pior_condicao_veiculo = df.loc[:, 'Vehicle_condition'].min()

print(f'A melhor condição de veículo é :{melhor_condicao_veiculo}')
print(f'A pior condição de veículo é :{pior_condicao_veiculo}')


A melhor condição de veículo é :3
A pior condição de veículo é :0


## A avaliação média por entregador.

In [10]:
columns = ['Delivery_person_Ratings','Delivery_person_ID']
avg_delivery_ratings = df1.loc[:, columns].groupby('Delivery_person_ID').mean()

avg_delivery_ratings

,Delivery_person_Ratings
Delivery_person_ID,
AGRRES010DEL01,4.761538
AGRRES010DEL02,4.671429
AGRRES010DEL03,4.575000
AGRRES01DEL01,4.522222
AGRRES01DEL02,4.700000
...,...
VADRES19DEL02,4.632727
VADRES19DEL03,4.670270
VADRES20DEL01,4.620370


## A avaliação média e o desvio padrão por tipo de tráfego.

In [11]:
columns = ['Road_traffic_density', 'Delivery_person_Ratings']


#avaliação média e desvio padrão 
avg_std_ratings_per_traffic = (df1.loc[:, columns].groupby(['Road_traffic_density'])
         .agg({'Delivery_person_Ratings': ['mean', 'std']}))



#nome colunas
avg_std_ratings_per_traffic.columns = ['Delivery_mean', 'Delivery_std']
avg_std_ratings_per_traffic = avg_std_ratings_per_traffic.reset_index()

avg_std_ratings_per_traffic

,Road_traffic_density,Delivery_mean,Delivery_std
0,High,4.652230,0.273044
1,Jam,4.594019,0.329778
2,Low,4.645011,0.338080
3,Medium,4.660138,0.274245


## A avaliação média e o desvio padrão por condições climáticas.

In [12]:
columns = ['Delivery_person_Ratings', 'Weatherconditions']


#avaliação média e desvio padrão 
avg_std_ratings_per_weather = (df1.loc[:, columns].groupby(['Weatherconditions'])
         .agg({'Delivery_person_Ratings': ['mean', 'std']}))



#nome colunas
avg_std_ratings_per_weather.columns = ['Delivery_mean', 'Delivery_std']
avg_std_ratings_per_weather = avg_std_ratings_per_weather.reset_index()

avg_std_ratings_per_weather

,Weatherconditions,Delivery_mean,Delivery_std
0,conditions Cloudy,4.651871,0.281197
1,conditions Fog,4.652965,0.275060
2,conditions Sandstorms,4.611748,0.310852
3,conditions Stormy,4.611819,0.313096
4,conditions Sunny,4.654868,0.396674
5,conditions Windy,4.616128,0.304565


## Os 10 entregadores mais rápidos por cidade.

In [13]:
cols = ['Delivery_person_ID', 'City', 'Time_taken(min)']
cities = df1['City'].unique()
df2 = df1.loc[:, cols].groupby(['City', 'Delivery_person_ID']).min().sort_values(['City', 'Time_taken(min)']).reset_index()

df_aux01 = df2.loc[df2['City'] == 'Urban'].head(10)
df_aux02 = df2.loc[df2['City'] == 'Metropolitian'].head(10)
df_aux03 = df2.loc[df2['City'] == 'Semi-Urban'].head(10)

df3 = pd.concat([df_aux01, df_aux02, df_aux03]).reset_index()

df3

,index,City,Delivery_person_ID,Time_taken(min)
0,1465,Urban,AGRRES03DEL01,10
1,1466,Urban,AGRRES08DEL03,10
2,1467,Urban,ALHRES01DEL01,10
3,1468,Urban,ALHRES04DEL01,10
4,1469,Urban,ALHRES19DEL01,10
5,1470,Urban,AURGRES02DEL03,10
6,1471,Urban,AURGRES08DEL02,10
7,1472,Urban,AURGRES11DEL02,10
8,1473,Urban,AURGRES15DEL02,10
9,1474,Urban,AURGRES16DEL01,10


## Os 10 entregadores mais lentos por cidade.

In [14]:
cols = ['Delivery_person_ID', 'City', 'Time_taken(min)']
cities = df1['City'].unique()
df2 = df1.loc[:, cols].groupby(['City', 'Delivery_person_ID']).max().sort_values(['City', 'Time_taken(min)'], ascending=False).reset_index()

df_aux01 = df2.loc[df2['City'] == 'Urban'].head(10)
df_aux02 = df2.loc[df2['City'] == 'Metropolitian'].head(10)
df_aux03 = df2.loc[df2['City'] == 'Semi-Urban'].head(10)

df3 = pd.concat([df_aux01, df_aux02, df_aux03]).reset_index()

df3


,index,City,Delivery_person_ID,Time_taken(min)
0,0,Urban,JAPRES05DEL02,54
1,1,Urban,JAPRES12DEL02,54
2,2,Urban,JAPRES17DEL01,54
3,3,Urban,PUNERES14DEL02,54
4,4,Urban,SURRES08DEL02,54
5,5,Urban,SURRES08DEL03,54
6,6,Urban,VADRES05DEL03,54
7,7,Urban,ALHRES09DEL02,53
8,8,Urban,CHENRES20DEL02,53
9,9,Urban,JAPRES03DEL01,53


# 3.0 Visão empresa


## A quantidade de entregadores únicos.

In [15]:
delivery_person_qty = len(df1.loc[:, 'Delivery_person_ID'].unique())

print(f'A quantidade de entregadores é: {delivery_person_qty}')

A quantidade de entregadores é: 1320


## A distância média dos resturantes e dos locais de entrega.

In [16]:
cols = ['Restaurant_latitude', 'Restaurant_longitude', 'Delivery_location_latitude', 'Delivery_location_longitude']

df1['delivery_distance'] = df1.loc[:, cols].apply(lambda x: 
                haversine( 
                    (x['Restaurant_latitude'], x['Restaurant_longitude']), 
                    (x['Delivery_location_latitude'], x['Delivery_location_longitude']))
                    ,axis=1)


avg_distance = df1.loc[:, ['City', 'delivery_distance']].groupby(['City']).mean().reset_index()


fig = go.Figure(data=[go.Pie(labels=avg_distance['City'], values=avg_distance['delivery_distance'], pull=[0,0.1,0])]) 

fig.show()

## O tempo médio e o desvio padrão de entrega por cidade

In [17]:
cols = ['Time_taken(min)','City']
avg_std_time_per_city = df1.loc[:, cols].groupby(['City']).agg({'Time_taken(min)': ['mean','std']})

avg_std_time_per_city.columns= ['avg_time', 'std_time']
avg_std_time_per_city = avg_std_time_per_city.reset_index()



avg_std_time_per_city
fig = go.Figure()
fig.add_trace(go.Bar(name='Control', x=avg_std_time_per_city['City'], y=avg_std_time_per_city['avg_time'], error_y=dict(type='data', array=avg_std_time_per_city['std_time'])))
fig.update_layout(barmode='group')

fig.show()

## O tempo médio e o desvio padrão de entrega por cidade e tipo de pedido.


In [18]:
cols = ['Time_taken(min)','City', 'Type_of_order']

avg_std_time_per_city_order_type = df1.loc[:, cols].groupby(['City','Type_of_order']).agg({'Time_taken(min)': ['mean','std']})
avg_std_time_per_city_order_type.columns= ['avg_time', 'std_time']
avg_std_time_per_city_order_type = avg_std_time_per_city_order_type.reset_index()

avg_std_time_per_city_order_type


,City,Type_of_order,avg_time,std_time
0,Metropolitian,Buffet,27.299008,9.153107
1,Metropolitian,Drinks,27.322691,9.041655
2,Metropolitian,Meal,27.616383,9.214536
3,Metropolitian,Snack,27.468414,9.119676
4,Semi-Urban,Buffet,49.707317,2.731702
5,Semi-Urban,Drinks,49.625000,2.459347
6,Semi-Urban,Meal,50.300000,3.041665
7,Semi-Urban,Snack,49.408163,2.707385
8,Urban,Buffet,23.560652,9.056348
9,Urban,Drinks,23.311977,8.927314


## O tempo médio e o desvio padrão de entrega por cidade e tipo de tráfego.


In [19]:
cols = ['Time_taken(min)','City', 'Road_traffic_density']
avg_std_time_per_city_traffic = df1.loc[:, cols].groupby(['City','Road_traffic_density']).agg({'Time_taken(min)': ['mean','std']})

avg_std_time_per_city_traffic.columns= ['avg_time', 'std_time']
avg_std_time_per_city_traffic = avg_std_time_per_city_traffic.reset_index()

fig = px.sunburst(avg_std_time_per_city_traffic, path=['City', 'Road_traffic_density'], values='avg_time',
                  color='std_time', color_continuous_scale='RdBu',
                  color_continuous_midpoint= np.mean(avg_std_time_per_city_traffic['std_time']))

fig.show()

##  O tempo médio de entrega durantes os Festivais.


In [20]:

cols = ['Festival', 'Time_taken(min)']

df3 = df1.loc[:, cols].groupby(['Festival']).mean().reset_index()
filtro =df3['Festival']== 'Yes'
festival_deliveries_time = df3.loc[filtro, 'Time_taken(min)']

festival_deliveries_time

1    45.518607
Name: Time_taken(min), dtype: float64